In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge  # スタッキング用
from sklearn.metrics import make_scorer

In [2]:
# nDCG@Kの計算
def ndcg_at_k(y_true, y_pred, k=22):
    def dcg(scores):
        return np.sum((2**scores - 1) / np.log2(np.arange(2, scores.size + 2)))

    k = min(k, len(y_true), len(y_pred))
    if k == 0:
        return 0
    
    order = np.argsort(y_pred)[::-1][:k]
    ideal_order = np.argsort(y_true)[::-1][:k]

    order = order[order < len(y_true)]
    ideal_order = ideal_order[ideal_order < len(y_true)]
    
    return dcg(y_true[order]) / (dcg(y_true[ideal_order]) + 1e-10)

In [3]:
# データ読み込み
train_df = pd.read_csv('../001.preprocess/train/train_A.csv')
test_df = pd.read_csv('../001.preprocess/test/test_A.csv')

In [4]:
# カテゴリ変数のエンコード
le_user = LabelEncoder()
le_product = LabelEncoder()
train_df["user_id"] = le_user.fit_transform(train_df["user_id"])
train_df["product_id"] = le_product.fit_transform(train_df["product_id"])
test_df["user_id"] = le_user.transform(test_df["user_id"])

In [5]:
# ランクのクリッピング
train_df["rank"] = train_df["rank"].clip(upper=31)

In [6]:
# 特徴量とターゲットの設定
X = train_df[["user_id", "product_id"]]
y = train_df["rank"]

In [7]:
# データの分割
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# ラベルが1から始まっているため、0から始まるように調整
y_train = y_train - 1
y_valid = y_valid - 1

In [9]:
# XGBoost用データフォーマットに変換
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

In [10]:
# LightGBMに必要なクエリ情報を作成
# 各ユーザーごとの商品数をカウントして、そのカウントをクエリ情報として使用
train_groups = X_train.groupby('user_id').size().values

In [11]:
# train_dfでの各ユーザーのグループサイズを取得
train_group_sizes = train_df.groupby('user_id').size()

In [12]:
# X_valid['user_id']に対して、train_dfのグループサイズをマッピング
valid_groups = X_valid['user_id'].map(train_group_sizes).fillna(0).astype(int)

In [13]:
# グループサイズの合計がX_validのサンプル数と一致することを確認
assert sum(valid_groups) == len(X_valid), "Valid group sum does not match number of samples"

AssertionError: Valid group sum does not match number of samples

In [14]:
print(sum(valid_groups))  # valid_groupsの合計
print(len(X_valid))  # X_validのサンプル数

49302794
295492


In [15]:
# valid_groupsの最初の数行を確認
print(valid_groups.head())
# X_validの各user_idとグループサイズを確認
print(X_valid[['user_id']].head())

1191955     28
1268961     53
104516      23
1320905     10
363296     140
Name: user_id, dtype: int64
         user_id
1191955    45922
1268961    48393
104516      3427
1320905    50018
363296     12030


In [16]:
# X_valid と valid_groups を照らし合わせて、グループサイズを確認
valid_groups_check = X_valid[['user_id']].copy()
valid_groups_check['group_size'] = valid_groups

In [17]:
# ユーザーごとのグループサイズの確認
print(valid_groups_check.head())

         user_id  group_size
1191955    45922          28
1268961    48393          53
104516      3427          23
1320905    50018          10
363296     12030         140


In [18]:
# X_validとvalid_groupsを統合してみる
valid_data_check = X_valid[['user_id']].copy()
valid_data_check['group_size'] = valid_groups

# 最初の数行を表示
print(valid_data_check.head())

         user_id  group_size
1191955    45922          28
1268961    48393          53
104516      3427          23
1320905    50018          10
363296     12030         140


In [19]:
# X_valid 内で各 user_id ごとのグループ数を計算
valid_groups_corrected = X_valid.groupby('user_id').size()

# 正しいグループサイズの確認
print(valid_groups_corrected.head())
print(sum(valid_groups_corrected))  # 合計を確認
print(len(X_valid))  # X_validのサンプル数を再確認

user_id
1    3
2    4
3    5
4    3
5    7
dtype: int64
295492
295492


In [20]:
# 各 user_id ごとのグループサイズを X_valid に適用
valid_groups = X_valid['user_id'].map(valid_groups_corrected)

# 再確認
print(valid_groups.head())
print(sum(valid_groups))  # 合計を確認
print(len(X_valid))  # X_validのサンプル数

1191955     6
1268961    15
104516      7
1320905     4
363296     25
Name: user_id, dtype: int64
10171390
295492


In [21]:
# 各 user_id ごとのグループサイズを X_valid に適用
valid_groups = X_valid['user_id'].map(valid_groups_corrected)

# 再確認
print(valid_groups.head())
print(f"Sum of valid_groups: {sum(valid_groups)}")  # 合計を確認
print(f"Length of X_valid: {len(X_valid)}")  # X_validのサンプル数

1191955     6
1268961    15
104516      7
1320905     4
363296     25
Name: user_id, dtype: int64
Sum of valid_groups: 10171390
Length of X_valid: 295492


In [22]:
# X_valid に存在しない user_id が含まれている場合、それを扱う方法を考慮します。
valid_groups = X_valid['user_id'].map(valid_groups_corrected).fillna(0).astype(int)

# 再確認
print(valid_groups.head())
print(f"Sum of valid_groups: {sum(valid_groups)}")  # 合計を確認
print(f"Length of X_valid: {len(X_valid)}")  # X_validのサンプル数

1191955     6
1268961    15
104516      7
1320905     4
363296     25
Name: user_id, dtype: int64
Sum of valid_groups: 10171390
Length of X_valid: 295492


In [23]:
# 存在しない user_id のリストを表示
missing_user_ids = X_valid[~X_valid['user_id'].isin(valid_groups_corrected.index)]
print(missing_user_ids)

Empty DataFrame
Columns: [user_id, product_id]
Index: []


In [24]:
# ユーザーごとのグループ数の確認
print(X_valid.groupby('user_id').size().head())  # user_id ごとのサイズを確認

user_id
1    3
2    4
3    5
4    3
5    7
dtype: int64


In [25]:
# user_id ごとのグループサイズを表示
print(valid_groups_corrected.head())
print(f"Sum of valid_groups_corrected: {valid_groups_corrected.sum()}")  # 合計確認

user_id
1    3
2    4
3    5
4    3
5    7
dtype: int64
Sum of valid_groups_corrected: 295492


In [26]:
# LightGBM用のデータフォーマットに変換
train_data_lgb = lgb.Dataset(X_train, label=y_train, group=train_groups)
valid_data_lgb = lgb.Dataset(X_valid, label=y_valid, group=valid_groups)

In [27]:
# 各モデルのパラメータ設定
params_xgb = {
    "objective": "rank:pairwise",
    "eval_metric": "ndcg",
    "tree_method": "hist",
    "max_depth": 6,
    "learning_rate": 0.1,
    "lambda": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 0.1
}

In [28]:
params_lgb = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "boosting_type": "gbdt",
    "max_depth": 6,
    "learning_rate": 0.1,
    "lambda_l2": 1.0,
    "subsample": 0.8, 
    "colsample_bytree": 0.8,
    "early_stopping_round": 10 
}

In [29]:
params_cb = {
    "loss_function": "YetiRank",
    "eval_metric": "NDCG",
    "depth": 6,
    "learning_rate": 0.1,
    "l2_leaf_reg": 1.0,
    "subsample": 0.8,
    "colsample_bylevel": 0.8
}

In [30]:
# モデルの学習
model_xgb = xgb.train(params_xgb, dtrain, num_boost_round=100, evals=[(dvalid, "valid")], early_stopping_rounds=10)
model_lgb = lgb.train(params_lgb, train_data_lgb, num_boost_round=100, valid_sets=[valid_data_lgb])
model_cb = cb.CatBoost(params_cb)
model_cb.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=10, verbose=False)

[0]	valid-ndcg:0.14174
[1]	valid-ndcg:0.27816
[2]	valid-ndcg:0.28412
[3]	valid-ndcg:0.41454
[4]	valid-ndcg:0.41454
[5]	valid-ndcg:0.41454
[6]	valid-ndcg:0.37899
[7]	valid-ndcg:0.37899
[8]	valid-ndcg:0.45475
[9]	valid-ndcg:0.38478
[10]	valid-ndcg:0.40479
[11]	valid-ndcg:0.26134
[12]	valid-ndcg:0.36720
[13]	valid-ndcg:0.22610
[14]	valid-ndcg:0.22610
[15]	valid-ndcg:0.22610
[16]	valid-ndcg:0.22610
[17]	valid-ndcg:0.22610
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to constrain model complexity.
[LightGBM] [Warning] Provided parameters constrain tree depth (max_depth=6) without explicitly setting 'num_leaves'. This can lead to underfitting. To resolve this warning, pass 'num_leaves' (<=64) in params. Alternatively, pass (max_depth=-1) and just use 'num_leaves' to cons

[LightGBM] [Fatal] Sum of query counts (295492) differs from the length of #data (10171390)


LightGBMError: Sum of query counts (295492) differs from the length of #data (10171390)

In [ ]:
print(sum(train_groups))  # train_groupsの合計
print(len(X_train))  # X_trainのサンプル数

print(sum(valid_groups))  # valid_groupsの合計
print(len(X_valid))  # X_validのサンプル数